# SuperMimic (`zita-ch/supermimic_vibed`) — Google Colab 一键全自动复现

本 Notebook 已将 [zita-ch/supermimic_vibed](https://github.com/zita-ch/supermimic_vibed)（论文 [SuperMimic: Scaling Humanoid Whole-Body Control with One Differentiable World Model](https://www.alphaxiv.org/abs/2610.28480) 的开源复现）完整适配为 **Google Colab 免 Docker 一键运行版本**。

### 使用方法
1. 点击顶部菜单 **Runtime（代码执行程序） -> Change runtime type（更改运行时类型）**，选择 **GPU**（推荐 **A100** 或 **L4** 并开启 **High-RAM**；免费版 **T4** 亦可自动降级参数运行）。
2. 点击 **Runtime -> Run all（全部运行）** 即可自动完成依赖安装、数据集下载与转换、自适应参数训练、曲线绘制及动作视频渲染。

In [ ]:
# @title 1. 检查 GPU 硬件并挂载 Google Drive（用于持久化保存 Checkpoint）
import os
import subprocess

USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
RUN_NAME = "lafan1_main"  # @param {type:"string"}

# 1. 检查 NVIDIA GPU
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"[GPU 已就绪] {gpu_info}")
except Exception as e:
    raise RuntimeError("未检测到 NVIDIA GPU！请在菜单 Runtime -> Change runtime type 中将硬件加速器切换为 GPU。") from e

# 2. 挂载 Google Drive 以防 Colab 断连丢失进度
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = f"/content/drive/MyDrive/supermimic_logs/{RUN_NAME}"
else:
    LOG_DIR = f"/content/supermimic_vibed/logs/{RUN_NAME}"

os.makedirs(LOG_DIR, exist_ok=True)
os.environ["SUPERMIMIC_LOG_DIR"] = LOG_DIR
print(f"[日志与权重保存路径] {LOG_DIR}")

In [ ]:
# @title 2. 安装系统依赖、配置 EGL 并部署 Isaac Lab 3 Kit-less 环境（约需 5-8 分钟）
%%bash
set -euo pipefail

WORKDIR="/content"
ISAACLAB_COMMIT="46d6cc3d7b115906fad12efd372fbf69f4e3d3ed"
VENV_DIR="/opt/isaaclab-venv"

echo "=== [1/3] 安装系统依赖与配置 NVIDIA EGL 离线渲染 ==="
apt-get update -qq && apt-get install -y -qq --no-install-recommends \
    tmux ffmpeg util-linux build-essential python3-dev cmake \
    libegl1 libgl1 libgl1-mesa-dev libx11-dev libxcursor-dev \
    libxi-dev libxinerama-dev libxrandr-dev curl git

cd "$WORKDIR"
if [ ! -d "$WORKDIR/supermimic_vibed" ]; then
    git clone https://github.com/zita-ch/supermimic_vibed.git
fi

mkdir -p /usr/share/glvnd/egl_vendor.d
cp "$WORKDIR/supermimic_vibed/docker/10_nvidia.json" /usr/share/glvnd/egl_vendor.d/10_nvidia.json

echo "=== [2/3] 安装 uv 并检出指定 Commit 的 Isaac Lab 3 ==="
if ! command -v uv &> /dev/null; then
    curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"

cd "$WORKDIR"
if [ ! -d "$WORKDIR/IsaacLab" ]; then
    git clone https://github.com/isaac-sim/IsaacLab.git
fi
git -C "$WORKDIR/IsaacLab" checkout "$ISAACLAB_COMMIT"

echo "=== [3/3] 构建 Python 3.12 虚拟环境并安装 Isaac Lab 3 Kit-less + Importers ==="
export UV_PROJECT_ENVIRONMENT="$VENV_DIR"
cd "$WORKDIR/IsaacLab"
uv sync --extra importers

# 自动检测驱动版本：若低于 580，切换为兼容 CUDA 12.4 的 PyTorch
DRIVER_MAJOR=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -n 1 | cut -d. -f1 || echo "0")
echo "当前 GPU 驱动主版本号: ${DRIVER_MAJOR}"
if [ "$DRIVER_MAJOR" -lt 580 ]; then
    echo "检测到驱动低于 580，正在切换为兼容 CUDA 12.4 的 PyTorch 构建..."
    uv pip install --python "$VENV_DIR/bin/python" -U torch torchvision --index-url https://download.pytorch.org/whl/cu124
fi
echo "环境安装完成！"

In [ ]:
# @title 3. 下载 LAFAN1 动作数据集与 G1 机器人模型并转换为 MJCF/USD 资产
%%bash
set -euo pipefail

export MUJOCO_GL=egl
export NVIDIA_DRIVER_CAPABILITIES=all
VENV_PY="/opt/isaaclab-venv/bin/python"

cd /content/supermimic_vibed
HF="https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset"
API="https://huggingface.co/api/datasets/lvhaidong/LAFAN1_Retargeting_Dataset/tree/main"
list_hf() {
    curl -s "$API/$1" | "$VENV_PY" -c "import json,sys; [print(x['path']) for x in json.load(sys.stdin)]"
}

mkdir -p data/lafan1_g1 data/g1_description/meshes
for f in $(list_hf g1); do
    curl -sfL -o "data/lafan1_g1/$(basename "$f")" "$HF/resolve/main/$f"
done
curl -sfL -o data/g1_description/g1_29dof_rev_1_0.urdf "$HF/resolve/main/robot_description/g1/g1_29dof_rev_1_0.urdf"
for f in $(list_hf robot_description/g1/meshes); do
    curl -sfL -o "data/g1_description/meshes/$(basename "$f")" "$HF/resolve/main/$f"
done
echo "已下载 G1 动作片段数量: $(ls data/lafan1_g1 | wc -l) clips"

echo "正在转换 URDF -> MJCF -> USD..."
"$VENV_PY" tools/urdf_to_mjcf.py
"$VENV_PY" tools/mjcf_to_usd.py

echo "正在运行环境与正运动学自检 (test_env.py)..."
"$VENV_PY" tools/test_env.py

In [ ]:
# @title 4. 启动自适应 GPU 参数训练（支持自动识别 A100 / L4 / T4 与断点续训）
import glob
import os
import subprocess

ITERATIONS = 14000  # @param {type:"integer"}
CKPT_EVERY = 2000   # @param {type:"integer"}
AUTO_RESUME = True  # @param {type:"boolean"}

log_dir = os.environ.get("SUPERMIMIC_LOG_DIR", "/content/supermimic_vibed/logs/lafan1_main")
os.makedirs(log_dir, exist_ok=True)

# 检测 GPU 显存与计算能力以自适应配置参数
gpu_query = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    text=True
).strip().split(",")
gpu_name = gpu_query[0].strip()
vram_mb = int(float(gpu_query[1].strip()))
compute_major = int(gpu_query[2].strip().split(".")[0])

if vram_mb >= 35000 and compute_major >= 8:
    # A100 / H100 等大显存 Ampere+ GPU
    num_envs, pi_batch, wm_batch, amp_mode = 4096, 4096, 2048, "bf16"
elif vram_mb >= 20000 and compute_major >= 8:
    # L4 (24GB) 等中大显存 Ada/Ampere GPU
    num_envs, pi_batch, wm_batch, amp_mode = 2048, 2048, 2048, "bf16"
else:
    # T4 (16GB, Turing SM 7.5) 等免费版 GPU
    num_envs, pi_batch, wm_batch, amp_mode = 1024, 1024, 1024, "none"

print(f"检测到 {gpu_name} ({vram_mb} MB, SM {compute_major}.x) -> 自动启用参数: "
      f"num_envs={num_envs}, pi_batch={pi_batch}, wm_batch={wm_batch}, amp={amp_mode}")

cmd = [
    "/opt/isaaclab-venv/bin/python", "supermimic/train.py",
    "--log_dir", log_dir,
    "--num_envs", str(num_envs),
    "--pi_batch", str(pi_batch),
    "--wm_batch", str(wm_batch),
    "--amp", amp_mode,
    "--iterations", str(ITERATIONS),
    "--ckpt_every", str(CKPT_EVERY),
]

# 查找是否已有 Checkpoint 可供续训
if AUTO_RESUME:
    ckpts = sorted(
        glob.glob(os.path.join(log_dir, "model_*.pt")),
        key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0])
    )
    if ckpts:
        latest_ckpt = ckpts[-1]
        print(f"发现已有权重，自动从 {latest_ckpt} 断点续训...")
        cmd.extend(["--resume", latest_ckpt])

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"
env["NVIDIA_DRIVER_CAPABILITIES"] = "all"

process = subprocess.Popen(
    cmd,
    cwd="/content/supermimic_vibed",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise RuntimeError(f"训练进程异常退出，返回码: {process.returncode}")

In [ ]:
# @title 5. 评估模型并渲染生成动作跟踪对比视频（直接在页面内播放）
import glob
import os
import subprocess
from base64 import b64encode
from IPython.display import HTML, Image, display

CLIP_NAME = "dance2_subject1"  # @param {type:"string"}
log_dir = os.environ.get("SUPERMIMIC_LOG_DIR", "/content/supermimic_vibed/logs/lafan1_main")
repo_dir = "/content/supermimic_vibed"
venv_py = "/opt/isaaclab-venv/bin/python"

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"

ckpts = sorted(
    glob.glob(os.path.join(log_dir, "model_*.pt")),
    key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0])
)
if not ckpts:
    raise FileNotFoundError(f"在 {log_dir} 中未找到任何 model_*.pt 权重文件，请先运行训练单元格。")

latest_ckpt = ckpts[-1]
print(f"使用最新权重进行评估与渲染: {latest_ckpt}")

traj_path = os.path.join(repo_dir, "traj.npz")
mp4_path = os.path.join(repo_dir, "out.mp4")

subprocess.run(
    [venv_py, "supermimic/evaluate.py", "--run_dir", log_dir, "--ckpt", latest_ckpt, "--save_traj", traj_path],
    cwd=repo_dir, env=env, check=True
)
subprocess.run(
    [venv_py, "tools/render.py", traj_path, mp4_path, "--clips", CLIP_NAME, "--second_view_clip", CLIP_NAME],
    cwd=repo_dir, env=env, check=True
)

if os.path.exists(mp4_path):
    mp4_data = open(mp4_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
    display(HTML(f'<video width=720 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))